# THINGS ResNet-50 Baseline on Colab

Open this notebook in VS Code and select a Google Colab GPU kernel. The notebook calls the project scripts; it does not duplicate training logic.

Training should run from `/content/human-things`, not directly from Google Drive, because THINGS has many small image files.

In [14]:
import sys
from pathlib import Path

try:
    import torch
    print('Python:', sys.version)
    print('Torch:', torch.__version__)
    print('CUDA:', torch.cuda.is_available())
    print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
except Exception as exc:
    print('Torch check failed:', repr(exc))

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA: True
Device: Tesla T4


## Configure

`REPO_URL` is set to the project GitHub repo. `DRIVE_DATA_ROOT` is the Drive folder that contains the project `data/` folder.

If the default path is wrong, run the Drive discovery cell below. It searches Drive for folders that look like this:

```text
SOME_FOLDER/data/processed/images.csv
SOME_FOLDER/data/raw/THINGS-database/...
```


In [15]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

REPO_URL = 'https://github.com/sabszh/human-things.git'

# Change this if your Drive folder is different. The folder must contain a data/ subfolder.
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/human-things-data')
PROJECT_ROOT = Path('/content/human-things')

print('Drive data root:', DRIVE_DATA_ROOT)
print('Expected data folder:', DRIVE_DATA_ROOT / 'data')
print('Project root:', PROJECT_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive data root: /content/drive/MyDrive/human-things-data
Expected data folder: /content/drive/MyDrive/human-things-data/data
Project root: /content/human-things


## Find Drive Data Folder

Run this if the default `DRIVE_DATA_ROOT` path does not exist. If candidates are found, set `DRIVE_DATA_ROOT` to one of them in the next cell.


In [16]:
from pathlib import Path

mydrive = Path('/content/drive/MyDrive')

processed_hits = sorted(mydrive.glob('**/data/processed/images.csv'))
raw_hits = sorted(mydrive.glob('**/data/raw/THINGS-database'))

candidate_roots = []
for hit in processed_hits:
    # .../ROOT/data/processed/images.csv -> ROOT
    candidate_roots.append(hit.parents[2])
for hit in raw_hits:
    # .../ROOT/data/raw/THINGS-database -> ROOT
    candidate_roots.append(hit.parents[2])

unique_candidates = []
for candidate in candidate_roots:
    if candidate not in unique_candidates:
        unique_candidates.append(candidate)

print('Candidate DRIVE_DATA_ROOT folders:')
for i, candidate in enumerate(unique_candidates):
    print(f'[{i}] {candidate}')

if not unique_candidates:
    print('No candidates found. Upload/copy your data folder to Drive, or set DRIVE_DATA_ROOT manually.')
else:
    print('To use a candidate, run for example:')
    print(f'DRIVE_DATA_ROOT = Path({str(unique_candidates[0])!r})')


Candidate DRIVE_DATA_ROOT folders:
No candidates found. Upload/copy your data folder to Drive, or set DRIVE_DATA_ROOT manually.


## Optional: Override Drive Data Root

Edit and run this only if the discovery cell found a better path.


In [17]:
# Example override. Uncomment and edit if needed.
# DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/YOUR_FOLDER_NAME')

print('Using DRIVE_DATA_ROOT:', DRIVE_DATA_ROOT)
print('Data folder exists:', (DRIVE_DATA_ROOT / 'data').exists())


Using DRIVE_DATA_ROOT: /content/drive/MyDrive/human-things-data
Data folder exists: False


## Clone or Update Repo

In [18]:
if not PROJECT_ROOT.exists():
    !git clone {REPO_URL} {PROJECT_ROOT}
else:
    %cd {PROJECT_ROOT}
    !git pull

%cd {PROJECT_ROOT}
!find scripts -maxdepth 1 -type f | sort

/content/human-things
Already up to date.
/content/human-things
scripts/01_make_metadata_csv.py
scripts/02_make_image_splits.py
scripts/03_train_resnet50_image_only.py
scripts/04_extract_resnet50_embeddings.py
scripts/05_evaluate_resnet50_embeddings.py


## Prepare Data on Colab

This cell first tries to copy `data/` from Drive. If that folder is missing, it downloads the required THINGS files from OSF and prepares `data/processed` locally. Set `DOWNLOAD_FROM_OSF = True` to allow the fallback download.

The image archives are large: about 5 GB plus 1.18 GB before extraction.


In [19]:
%cd {PROJECT_ROOT}

source_data = DRIVE_DATA_ROOT / 'data'
target_data = PROJECT_ROOT / 'data'
DOWNLOAD_FROM_OSF = True
COPY_PREPARED_DATA_BACK_TO_DRIVE = False

print('Source data:', source_data)
print('Target data:', target_data)

if source_data.exists():
    !mkdir -p {target_data}
    !rsync -a --info=progress2 {source_data}/ {target_data}/
else:
    print('Drive data folder not found; using OSF setup fallback.')
    print('If this is not intended, run the Drive discovery cell above and set DRIVE_DATA_ROOT.')
    if not DOWNLOAD_FROM_OSF:
        raise FileNotFoundError(f'Drive data folder not found: {source_data}')
    !python -m pip install -q -r requirements.txt
    !python scripts/00_setup_things_data.py --download-images
    if COPY_PREPARED_DATA_BACK_TO_DRIVE:
        !mkdir -p {source_data}
        !rsync -a --info=progress2 data/ {source_data}/

!find data -maxdepth 3 -type f | sort | head -60
!echo THINGS image count:
!find data/raw/THINGS-database/osfstorage/images_THINGS/object_images -type f -name '*.jpg' | wc -l
!echo THINGSplus CC0 image count:
!find data/raw/THINGS-database/osfstorage/images_THINGSplus-CC0/object_images_CC0 -maxdepth 1 -type f -name '*.jpg' | wc -l


/content/human-things
Source data: /content/drive/MyDrive/human-things-data/data
Target data: /content/human-things/data
Drive data folder not found; using OSF setup fallback.
If this is not intended, run the Drive discovery cell above and set DRIVE_DATA_ROOT.
python3: can't open file '/content/human-things/scripts/00_setup_things_data.py': [Errno 2] No such file or directory
find: ‘data’: No such file or directory
THINGS image count:
find: ‘data/raw/THINGS-database/osfstorage/images_THINGS/object_images’: No such file or directory
0
THINGSplus CC0 image count:
find: ‘data/raw/THINGS-database/osfstorage/images_THINGSplus-CC0/object_images_CC0’: No such file or directory
0


## Install Requirements and Check GPU

In [ ]:
%cd {PROJECT_ROOT}
!python -m pip install -q -r requirements.txt

import torch, torchvision
print('torch', torch.__version__)
print('torchvision', torchvision.__version__)
print('cuda', torch.cuda.is_available())
print('device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Rebuild Metadata and Splits

In [ ]:
%cd {PROJECT_ROOT}
!python scripts/01_make_metadata_csv.py
!python scripts/02_make_image_splits.py
!cat outputs/image_metadata_report.json
!cat outputs/image_splits_report.json

## Dry Run Data Loading

In [ ]:
%cd {PROJECT_ROOT}
!python scripts/03_train_resnet50_image_only.py --dry-run --batch-size 16 --num-workers 2

## Train

Start with the short run. Only set `RUN_FULL = True` after the short run finishes cleanly.

In [ ]:
%cd {PROJECT_ROOT}

RUN_SHORT = True
RUN_FULL = False
BATCH_SIZE = 64
NUM_WORKERS = 2

if RUN_SHORT:
    !python scripts/03_train_resnet50_image_only.py --head-epochs 1 --layer4-epochs 1 --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS}

if RUN_FULL:
    !python scripts/03_train_resnet50_image_only.py --head-epochs 5 --layer4-epochs 10 --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS}

## Inspect Training Outputs

In [ ]:
%cd {PROJECT_ROOT}
!find outputs/baseline_resnet50 -maxdepth 2 -type f | sort || true
!test -f outputs/baseline_resnet50/metrics.json && cat outputs/baseline_resnet50/metrics.json || true
!test -f outputs/baseline_resnet50/training_log.csv && head -20 outputs/baseline_resnet50/training_log.csv || true

## Extract and Evaluate Embeddings

Run after `outputs/baseline_resnet50/best_model.pt` exists.

In [ ]:
%cd {PROJECT_ROOT}

RUN_EXTRACT_AND_EVAL = False

if RUN_EXTRACT_AND_EVAL:
    !python scripts/04_extract_resnet50_embeddings.py --batch-size 128 --num-workers 2
    !python scripts/05_evaluate_resnet50_embeddings.py
    !cat outputs/baseline_resnet50/embedding_eval_report.json

## Copy Results Back to Drive

In [ ]:
RUN_COPY_BACK = False
DRIVE_OUTPUT_ROOT = DRIVE_DATA_ROOT / 'outputs'

if RUN_COPY_BACK:
    !mkdir -p {DRIVE_OUTPUT_ROOT}
    !rsync -a --info=progress2 outputs/baseline_resnet50/ {DRIVE_OUTPUT_ROOT}/baseline_resnet50/
    !find {DRIVE_OUTPUT_ROOT}/baseline_resnet50 -maxdepth 3 -type f | sort